# Test 2: Qwen3 de 4B y 8B

**Resultado:** ambos tamaños completan el bucle en local, pero ninguna propuesta supera la validación. 8B cabe en la Mac; esta prueba todavía no justifica reemplazar 4B para continuar el estudio.

Se separa la decisión del mensaje y se evita reevaluar políticas repetidas. Es un ajuste manual del harness, no una mejora que el modelo haya descubierto.

Este notebook lee evidencia guardada. **Ejecutar sus celdas no carga modelos ni llama a una API.**

[Motor de la prueba](utils/local_model_test.py) · [Seis casos nuevos](local-transfer-cases.yaml) · [Ejecuciones anteriores y actuales](results/)

In [1]:
from pathlib import Path
import hashlib
import html
import json
from IPython.display import HTML, display

case_dir = Path.cwd()
if not (case_dir / "results").is_dir():
    case_dir = case_dir / "research/cases/case-0003-debt-negotiation"
run_names = [
    "20261031T054930Z-local-qwen3-decision-fixed",
    "20261031T055008Z-local-qwen3-decision-adaptive",
    "20261031T055055Z-local-qwen3-8b-decision-fixed",
    "20261031T055221Z-local-qwen3-8b-decision-adaptive",
]

def load_record(name):
    folder = case_dir / "results" / name
    manifest = json.loads((folder / "manifest.json").read_text())
    for filename, expected in manifest.items():
        assert Path(filename).name == filename
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == expected
    return {"name": name,
        "protocol": json.loads((folder / "protocol.json").read_text()),
        "report": json.loads((folder / "report.json").read_text()),
        "calls": [json.loads(line) for line in (folder / "calls.jsonl").read_text().splitlines()]}

records = [load_record(name) for name in run_names]

def label(record):
    size = "8B" if record["report"]["model_key"] == "qwen3-8b" else "4B"
    mode = "B1 fija" if record["protocol"]["strategy_mode"] == "fixed" else "B2 adaptativa"
    return f"{size} · {mode}"

def table(headers, rows):
    head = "".join(f'<th style="text-align:left;padding:8px;border-bottom:2px solid #667085">{html.escape(str(h))}</th>' for h in headers)
    body = "".join("<tr>" + "".join(f'<td style="text-align:left;vertical-align:top;padding:8px;border-bottom:1px solid #ddd">{html.escape(str(v))}</td>' for v in row) + "</tr>" for row in rows)
    display(HTML(f'<table style="width:100%;border-collapse:collapse;margin:16px 0"><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

def panel(title, text):
    display(HTML(f'<h4>{html.escape(title)}</h4><blockquote style="border-left:3px solid #667085;padding:12px;white-space:pre-wrap;overflow-wrap:anywhere">{html.escape(text)}</blockquote>'))

print("Integridad de las cuatro ejecuciones verificada.")

Integridad de las cuatro ejecuciones verificada.


## 1. Condiciones

- Mac M5 Max, 48 GB de memoria unificada. MLX, pesos a 4 bits, sin adaptadores ni entrenamiento.
- Temperatura 0, semilla 42 y `enable_thinking=False`. Un modelo por proceso.
- Tres propuestas por condición; máximo 35 llamadas, 50.000 tokens, 15 minutos y 12 GiB de asignación MLX.
- Máximo 1.024 tokens de salida; entrada más salida reservada: 4.096 tokens.
- Dos casos de desarrollo, dos de validación y dos de test ya conocidos.
- Seis casos nuevos preparados antes de las cuatro ejecuciones, consultados solo después de seleccionar la política. No se usan para proponer ni elegir candidatos.

Las cuatro condiciones comparten el protocolo v5. Cambian el tamaño del modelo y la actualización de las instrucciones de mejora. El presupuesto máximo es común; el consumo real puede ser distinto.

## 2. Qué mejora y qué permanece fijo

$H$ es la política del negociador. $\phi$ son las instrucciones del mejorador. $L$ es el modelo con pesos fijos.

$H'_t=M_{\phi_t}(H_t,O,F_{dev},historial;L)$: el mejorador propone una política desde el objetivo $O$, los errores de desarrollo y los dos intentos recientes.

En **B1**, φ permanece fijo. En **B2**, otra llamada actualiza φ con la política candidata y sus errores de desarrollo. El evaluador de Python permanece fijo en ambos casos.

**Flujo de respuesta:** política H + caso → LLM → decisión JSON → plantilla de mensaje → evaluación.

El LLM devuelve `action`, `debt`, `payments` e `installment`. Python genera `message` desde esos valores. No cambia acciones, montos ni planes incorrectos; el evaluador los rechaza.

In [2]:
table(["Parte", "Responsabilidad", "Editable por el modelo"], [
    ["Negociador", "Decidir la acción y el plan", "Política H"],
    ["Mejorador", "Proponer otra política usando errores e intentos recientes", "Instrucciones φ en B2"],
    ["Actualizador", "Proponer las instrucciones de la siguiente iteración", "Solo el texto φ"],
    ["Renderizador", "Redactar un mensaje desde los campos devueltos", "No"],
    ["Evaluador", "Comprobar decisión y restricciones; seleccionar", "No"],
])
panel("Contrato fijo del negociador", records[0]["protocol"]["runtime"])

Parte,Responsabilidad,Editable por el modelo
Negociador,Decidir la acción y el plan,Política H
Mejorador,Proponer otra política usando errores e intentos recientes,Instrucciones φ en B2
Actualizador,Proponer las instrucciones de la siguiente iteración,Solo el texto φ
Renderizador,Redactar un mensaje desde los campos devueltos,No
Evaluador,Comprobar decisión y restricciones; seleccionar,No


## 3. Resultado

$Q$ es la proporción de respuestas que coinciden con la acción y el plan esperados y pasan las reglas estructuradas. Se acepta una propuesta si mejora Q en validación y no presenta incumplimientos.

Una política duplicada se registra, pero reutiliza la evaluación anterior y no puede aceptarse como mejora nueva. La igualdad se comprueba normalizando Unicode y espacios; no se detectan paráfrasis equivalentes.

In [3]:
quality_rows, resource_rows = [], []
for record in records:
    r = record["report"]
    selected = next(item for item in r["rounds"] if item["iteration"] == r["selected_iteration"])
    duplicates = sum(item.get("duplicate_of") is not None for item in r["rounds"][1:])
    quality_rows.append([label(record), len(r["rounds"]) - 1 - duplicates, duplicates,
        sum(item["accepted"] for item in r["rounds"][1:]),
        f'{selected["validation"]["q"] * 2:.0f}/2',
        f'{r["transfer"]["initial"]["q"] * 6:.0f}/6',
        f'{r["transfer"]["selected"]["q"] * 6:.0f}/6'])
    resource_rows.append([label(record), r["calls"], r["tokens"],
        f'{r["elapsed_seconds"]:.1f}', f'{r["peak_mlx_gb"]:.2f}'])
table(["Condición", "Propuestas nuevas", "Repetidas", "Aceptadas", "Validación", "Nuevos H0", "Nuevos Hfinal"], quality_rows)
table(["Condición", "Llamadas", "Tokens totales", "Segundos", "Pico MLX (GB)"], resource_rows)

Condición,Propuestas nuevas,Repetidas,Aceptadas,Validación,Nuevos H0,Nuevos Hfinal
4B · B1 fija,1,2,0,0/2,0/6,0/6
4B · B2 adaptativa,2,1,0,0/2,0/6,0/6
8B · B1 fija,1,2,0,0/2,2/6,2/6
8B · B2 adaptativa,1,2,0,0/2,2/6,2/6


Condición,Llamadas,Tokens totales,Segundos,Pico MLX (GB)
4B · B1 fija,19,11288,14.1,3.18
4B · B2 adaptativa,25,16098,19.4,3.22
8B · B1 fija,19,11578,42.2,5.61
8B · B2 adaptativa,21,13868,47.0,5.66


Las cuatro condiciones conservaron **H0**. El 2/6 de 8B en los casos nuevos pertenece a esa política inicial, no a una política mejorada. El test reutilizado de dos casos dio 1/2 en las cuatro condiciones.

8B consumió unos 5,6 GB de memoria MLX frente a 3,2 GB de 4B. Los tiempos incluyen carga y ejecución, no descarga. Son orientativos: una ejecución por condición, con 4B antes de 8B. El pico MLX no equivale a toda la RAM del proceso.

El renderizador evita que un texto generado contradiga una decisión admisible, pero no resuelve errores de decisión. El resultado del modelo sigue siendo visible y no se corrige después.

## 4. Qué se evita repetir

El mejorador recibe los dos intentos recientes y sus fallos de desarrollo. Si vuelve a proponer el mismo texto, no se repiten sus cuatro evaluaciones: dos de desarrollo y dos de validación. La llamada que produjo el duplicado sí se cuenta.

In [4]:
rows = []
for record in records:
    for item in record["report"]["rounds"][1:]:
        rows.append([label(record), f'H{item["iteration"]}',
            f'H{item["parent_iteration"]}',
            "Nueva" if item["duplicate_of"] is None else f'Repite H{item["duplicate_of"]}',
            item["policy_tokens"], item["strategy_tokens"]])
table(["Condición", "Propuesta", "Desde", "Identidad", "Tokens H", "Tokens φ usados"], rows)
count = sum(item.get("duplicate_of") is not None for r in records for item in r["report"]["rounds"][1:])
print(f"{count} propuestas duplicadas detectadas; {4 * count} llamadas de evaluación omitidas.")

Condición,Propuesta,Desde,Identidad,Tokens H,Tokens φ usados
4B · B1 fija,H1,H0,Nueva,107,31
4B · B1 fija,H2,H0,Repite H1,107,31
4B · B1 fija,H3,H0,Repite H1,107,31
4B · B2 adaptativa,H1,H0,Nueva,107,31
4B · B2 adaptativa,H2,H0,Nueva,121,123
4B · B2 adaptativa,H3,H0,Repite H2,121,123
8B · B1 fija,H1,H0,Nueva,121,31
8B · B1 fija,H2,H0,Repite H1,121,31
8B · B1 fija,H3,H0,Repite H1,121,31
8B · B2 adaptativa,H1,H0,Nueva,121,31


7 propuestas duplicadas detectadas; 28 llamadas de evaluación omitidas.


Se detectaron siete propuestas repetidas. El control ahorró sus 28 reevaluaciones, pero el modelo todavía repite algunas propuestas pese a recibir el historial. No se afirma que haya aumentado la diversidad semántica.

La longitud se mide con el tokenizer nativo, sin tokens especiales: $\ell_H=|Tokenizer(H)|$ y $\ell_\phi=|Tokenizer(\phi)|$. Los tokens totales incluyen la plantilla de chat, contratos, datos y salidas. Esta medida registra tamaño y consumo; no mide claridad ni explica por sí sola la calidad.

## 5. Decisión original y mensaje construido

Este ejemplo conserva los valores del modelo. Un mensaje coherente con esos valores no vuelve correcta una decisión equivocada.

In [5]:
record = next(r for r in records if r["report"]["model_key"] == "qwen3-8b" and r["protocol"]["strategy_mode"] == "fixed")
item = record["report"]["rounds"][1]
row = item["validation"]["rows"][0]
panel("Política H1 propuesta por 8B, rechazada", item["policy"])
panel("Contexto visible", json.dumps(row["context"], ensure_ascii=False, indent=2))
panel("Decisión devuelta por el LLM", json.dumps(row["raw_reply"], ensure_ascii=False, indent=2))
panel("Mensaje generado por la plantilla", row["reply"]["message"])
table(["Comprobación", "Resultado"], [
    ["Decisión esperada", json.dumps(row["expected"], ensure_ascii=False)],
    ["Coincide con la tarea", "Sí" if row["task_match"] else "No"],
    ["Incumplimientos detectados", ", ".join(row["violations"]) or "Ninguno"],
    ["Respuesta correcta", "Sí" if row["success"] else "No"],
])

Comprobación,Resultado
Decisión esperada,"{""action"": ""offer"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100}"
Coincide con la tarea,No
Incumplimientos detectados,Ninguno
Respuesta correcta,No


## 6. Casos nuevos tras la selección

Estos seis casos se prepararon antes de ejecutar el contraste. No se enviaron al mejorador ni al actualizador, y no participaron en la selección. Son casos sintéticos diseñados manualmente, no una muestra representativa de conversaciones reales.

In [6]:
rows = []
for index, case in enumerate(records[0]["protocol"]["cases"]["transfer"]):
    expected = case["expected"]
    plan = f'{expected["action"]} / {expected["debt"]} / {expected["payments"]} / {expected["installment"]}'
    outcomes = ["Correcta" if r["report"]["transfer"]["selected"]["rows"][index]["success"] else "Incorrecta" for r in records]
    rows.append([case["id"].replace("transfer-", ""), plan, *outcomes])
table(["Caso", "Esperado: acción/deuda/pagos/cuota", *[label(r) for r in records]], rows)

Caso,Esperado: acción/deuda/pagos/cuota,4B · B1 fija,4B · B2 adaptativa,8B · B1 fija,8B · B2 adaptativa
unknown,ask / None / 0 / 0,Incorrecta,Incorrecta,Correcta,Correcta
capacity-150,offer / B / 2 / 150,Incorrecta,Incorrecta,Incorrecta,Incorrecta
capacity-300,offer / C / 2 / 300,Incorrecta,Incorrecta,Incorrecta,Incorrecta
capacity-600,offer / C / 1 / 600,Incorrecta,Incorrecta,Incorrecta,Incorrecta
declined,stop / None / 0 / 0,Incorrecta,Incorrecta,Correcta,Correcta
insufficient,stop / None / 0 / 0,Incorrecta,Incorrecta,Incorrecta,Incorrecta


## 7. Ejecuciones anteriores

Se conservan los registros anteriores, incluidos fallos de transporte y pruebas con contexto incompleto. El protocolo v4 corrigió la ausencia del historial en el feedback. V5 añade renderizado de mensajes, memoria de intentos, detección de duplicados y casos finales nuevos.

No se atribuye el cambio entre v4 y v5 a un componente aislado: se modificaron varias partes. El contraste 4B/8B de arriba comparte v5.

In [7]:
previous = []
for folder in sorted((case_dir / "results").glob("*-local-*")):
    if folder.name in run_names:
        continue
    record = load_record(folder.name)
    r, p = record["report"], record["protocol"]
    previous.append([folder.name, p["protocol"].rsplit("-", 1)[-1],
        "Completa" if r["status"] == "completed" else "Detenida", len(r["rounds"]) - 1 if r["rounds"] else 0])
table(["Registro previo", "Protocolo", "Estado", "Propuestas evaluadas"], previous)

Registro previo,Protocolo,Estado,Propuestas evaluadas
20261031T052541Z-local-qwen,v1,Detenida,0
20261031T052617Z-local-qwen,v2,Detenida,2
20261031T052638Z-local-gemma,v2,Detenida,0
20261031T052812Z-local-qwen3,v2,Completa,3
20261031T053709Z-local-qwen3-calibration,v3,Completa,0
20261031T053730Z-local-qwen3-explicit-fixed,v3,Completa,3
20261031T053758Z-local-qwen3-explicit-adaptive,v3,Completa,3
20261031T053849Z-local-qwen3-explicit-fixed,v4,Completa,3
20261031T053935Z-local-qwen3-explicit-adaptive,v4,Completa,3


## 8. Decisión de trabajo

- **Mantener 4B para depurar el harness y 8B como contraste.** Ambos ejecutan el bucle en local; 8B no consiguió una política aceptada y necesita más memoria y tiempo.
- Antes de subir a modelos mayores, conviene comprobar una política de referencia que resuelva la tarea y separar los fallos del negociador de los del mejorador.
- No se obtuvo evidencia de ventaja de la adaptación. Son tres propuestas por condición y pocos escenarios; tampoco se midió generalización ni importancia causal de longitud o formato.
- No hubo entrenamiento, llamadas a API ni contratación de GPU. Los pesos permanecen en local.

## Ejecutar el protocolo actual

Python 3.12 en Apple Silicon, con [requirements-local.txt](requirements-local.txt) y los snapshots fijados en la caché de Hugging Face:

```sh
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3 --contract decision --strategy fixed
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3 --contract decision --strategy adaptive
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3-8b --contract decision --strategy fixed
HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 python utils/local_model_test.py qwen3-8b --contract decision --strategy adaptive
```

Ejecutar uno a la vez. Cada ejecución genera un directorio nuevo con protocolo, respuestas, reporte, script y hashes. Este notebook muestra los registros fijados al inicio.

Fuentes de los modelos: [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) y [Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B). Las revisiones de las conversiones MLX utilizadas figuran en cada protocolo.